# Alexandria — rode o projeto inteiro aqui no navegador

Este notebook clona o repositório, instala o dbt, constrói as três camadas e executa os **75 testes de qualidade** — do zero, na máquina do Google, em cerca de dois minutos.

Não precisa instalar nada e não precisa acreditar em mim: o resultado que aparecer abaixo é o que acabou de rodar agora.

**Como usar:** menu `Ambiente de execução` → `Executar tudo`.

---

Repositório: https://github.com/GabiGattiRodrigues/alexandria

## 1. Instalação e clone

In [ ]:
!pip install -q dbt-duckdb==1.11.0
!git clone -q https://github.com/GabiGattiRodrigues/alexandria.git
%cd alexandria
!dbt deps

## 2. Dados

O projeto roda tanto sobre o dataset público da Olist quanto sobre uma amostra sintética com o mesmo schema, gerada por script. Aqui usamos a amostra, para o notebook não depender de credencial do Kaggle.

In [ ]:
!python scripts/gerar_amostra.py --destino data/sample --pedidos 4000
!python scripts/carregar_bronze.py --origem data/sample

## 3. Construção das camadas e execução dos testes

`dbt build` constrói bronze → silver → gold e roda cada teste no ponto certo do grafo: se um teste da silver falhar, os modelos gold que dependem dela nem chegam a ser construídos.

In [ ]:
!dbt build --profiles-dir .

## 4. Placar dos testes

O resumo abaixo é lido de `target/run_results.json`, o artefato que o próprio dbt acabou de escrever — nenhum número é digitado à mão.

In [ ]:
!python scripts/resumo_testes.py

import json
import pandas as pd

resumo = json.load(open('site/resultados.json'))
print('Status do build:', resumo['build']['status'].upper())
print(f"{resumo['build']['testes_aprovados']} de {resumo['build']['testes_total']} testes aprovados "
      f"em {resumo['build']['duracao_total_s']}s")

pd.DataFrame(
    resumo['por_tipo'].items(), columns=['tipo de teste', 'quantidade']
).set_index('tipo de teste')

In [ ]:
ax = pd.Series(resumo['por_camada']).plot.barh(
    figsize=(7, 2.6), color=['#A8703A', '#6E829A', '#9C7A10'], width=.7
)
ax.set_title('Testes por camada', loc='left', fontsize=11)
ax.set_xlabel('testes'); ax.spines[['top', 'right']].set_visible(False)
for i, v in enumerate(resumo['por_camada'].values()):
    ax.text(v + .5, i, str(v), va='center', fontsize=9)

## 5. O teste que sustenta a promessa do projeto

Dos 75, três são testes de reconciliação entre camadas. Este compara a receita e a contagem de pedidos do agregado diário — o que o dashboard desenha — contra a tabela-fato que o agente conversacional consulta. Se um `join` duplicar uma linha, a diferença aparece aqui, e não na frente de um executivo.

In [ ]:
print(open('tests/assert_kpis_diarios_reconciliam.sql').read())

## 6. Provando na unha

Consultando o DuckDB diretamente, sem passar pelo dbt: os dois caminhos têm que devolver o mesmo número.

In [ ]:
import duckdb

con = duckdb.connect('alexandria.duckdb')
con.sql('''
    select
        (select sum(pedidos)            from gold.agg_kpis_diarios) as pedidos_no_agregado,
        (select count(distinct pedido_id) from gold.fct_pedidos)    as pedidos_na_fato,
        (select round(sum(receita_bruta), 2) from gold.agg_kpis_diarios) as receita_no_agregado,
        (select round(sum(receita_bruta), 2) from gold.fct_pedidos)      as receita_na_fato
''').show()

## 7. Um exemplo do que a camada gold entrega

Segmentação RFV: quintis calculados sobre a base inteira na data de corte, com os oito rótulos definidos no modelo — e não em cada ferramenta de BI.

In [ ]:
con.sql('''
    select
        segmento_rfv,
        count(*)                       as clientes,
        round(avg(receita_total), 2)   as receita_media,
        round(avg(recencia_dias))      as recencia_media_dias
    from gold.agg_clientes_rfv
    group by 1
    order by receita_media desc
''').df()

## 8. E se um dado chegar errado?

A prova mais honesta de que o teste funciona é vê-lo falhar. Abaixo, duplicamos uma linha na camada bronze de propósito e rodamos os testes de novo.

In [ ]:
con.sql('insert into bronze.pedido_itens select * from bronze.pedido_itens limit 1')
con.close()

print('Uma linha duplicada na bronze. Rodando os testes de novo...\n')

In [ ]:
!dbt test --profiles-dir . --select fct_pedidos agg_kpis_diarios

O build quebra e diz exatamente onde: a receita da gold deixou de bater com a da silver.

É esse comportamento que faz a diferença entre "o dashboard está no ar" e "o número do dashboard é confiável".

---

[Repositório](https://github.com/GabiGattiRodrigues/alexandria) · [Documentação e linhagem](https://gabigattirodrigues.github.io/alexandria/) · [Dashboard Inteligente](https://dashboardinteligente.streamlit.app) · [LinkedIn](https://www.linkedin.com/in/gabriela-gatti-rodrigues)